# Session 2 — CI/CD for ML Systems

**Goal:** Understand how the manual workflow from Session 1 becomes an automated pipeline.

In Session 1 you manually:
1. Ran `04_mlflow_experiment.ipynb` → compared models
2. Ran `05_register_and_serve.ipynb` → deployed the best one

In production, this **must** happen automatically when:
- New training data arrives
- Code is merged to the main branch
- A scheduled retrain job triggers

**The tools:** Declarative Automation Bundles (DABs) + GitHub Actions

## Declarative Automation Bundles
Formerly known as Databricks Asset Bundles, facilitate the adoption of software engineering best practices, including source control, code review, testing, and continuous integration and delivery (CI/CD)

Declarative Automation Bundles include:
-  Infrastructure and workspace configurations
-  Source files
-  Definitionsfor Databricks resources (i.e. Lakeflow Jobs, Lakeflow Spark Declarative Pipelines, Dashboards, Model Serving endpoints, MLflow Experiments, and MLflow registered models)
-  Unit tests and integration tests

<img src="../utils/resources/bundles-cicd.png">

Declarative Automation Bundles are typically developed in a local development environment such as VSCode or Cursor.  Databricks also supports [DABs in the Workspace](https://www.databricks.com/blog/announcing-databricks-asset-bundles-now-workspace) which enables you to build, edit and deploy using DABs from within a Databricks workspace.  We will use DABs in the Workspace in this workshop so we won't need a local dev environment.

## Navigate the Declarative Automation Bundle
1. Navigate to `../bundle` the **Workspace Navigator**.
2. Open `databricks.yml`.  This is the root definition file of your bundle.
3. Open the `bundle/resources/retrain_job.yml`.  This defines a Lakeflow Job that runs multiple notebook tasks that train and deploy a new model, automating the tasks we performed manually in Session 1.

TODO: walkthrough of the DAB

## Deploy the bundle and run the training job

TODO: These instructions need more detail.

also

TODO: **You will have to change the dev and prod workspace host values to whatever workspace you are working in if it isn't the final workshop workspace.**

1. Select the "rocket" icon in the left navigator.  The bundle manager view opens.
2. Select **dev** target
2. **kebob** menu, then **Configure variable overrides**
```
{
  "schema" : "<safe_username>"
}
```
4. Click **Deploy**, then **Deploy** again to deploy the notebooks and jobs.
5. View the jobs:
   1.  Open **Jobs** from the left sidebar in a new browser tab
   2.  Note the new jobs that begin with `[dev <safe_username>]`.  The prefix is added automatically when deploying a DAB in a development target to enable parallel development.
   3.  Click the **Run now** icon next to the `[dev <safe_username>] workshop_retrain_churn` job to start the job.

   This job will take several minutes to complete.

## The Retrain Pipeline DAG

The retrain job has 4 tasks with dependencies:

```
Task 1: ingest_and_validate
    ↓  (fails if data quality checks fail)
Task 2: train_models
    ↓  (logs 3 MLflow runs, picks best)
Task 3: gate_and_register
    ↓  (fails if F1 drops > 5%)
Task 4: update_endpoint
```

This design ensures:
- **Bad data never triggers training** (Task 1 gate)
- **Worse models are never deployed** (Task 3 gate)
- **Every deployment is traceable** (MLflow + UC lineage)

TODO: navigate the DAB resource files in the UI, not copies in the notebook.

In [0]:
# Review the retrain job definition
# In a real project, this file lives at resources/retrain_job.yml

retrain_job_yaml = """
resources:
  jobs:
    workshop_retrain_churn:
      name: workshop_retrain_churn
      tasks:
        - task_key: ingest_and_validate
          notebook_task:
            notebook_path: ../src/jobs/01_ingest_and_validate.ipynb
          ...
        - task_key: train_models
          depends_on: [{task_key: ingest_and_validate}]
          ...
        - task_key: gate_and_register
          depends_on: [{task_key: train_models}]
          ...
        - task_key: update_endpoint
          depends_on: [{task_key: gate_and_register}]
"""
print(retrain_job_yaml)

## The GitHub Actions Workflow

The CI/CD pipeline has two triggers:

**On Pull Request** (`.github/workflows/pr_validation.yml`):
```
push to PR branch → run pytest → databricks bundle validate
```
Prevents merging broken code.

**On Merge to Main** (`.github/workflows/deploy_retrain.yml`):
```
merge to main → bundle deploy → bundle run retrain_job
```
Automatically deploys and retrains after every approved code change.

In [0]:
# Review the GitHub Actions workflow
pr_workflow = """
name: PR Validation
on:
  pull_request:
    branches: [main]

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main

      - name: Run unit tests
        run: |
          pip install pytest scikit-learn mlflow pyyaml
          pytest src/tests/ -v

      - name: Validate bundle
        run: databricks bundle validate
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
"""
print(pr_workflow)

## Discussion

- Where should the `DATABRICKS_TOKEN` come from? (Service principal, not a human user)
- What happens if the test fails? (PR is blocked from merging)
- What's the difference between `bundle validate` and `bundle deploy`?

➡️ Next: `02_trigger_retrain.ipynb` — manually trigger the retrain pipeline